# Primary Seven-Model Paired Comparison

## BRFSS 2025 Diabetes Classification

This notebook performs **post-hoc comparison only**.

All seven model configurations were frozen before this comparison. No hyperparameter, threshold, feature, preprocessing rule, or target definition is modified here.

### Why paired comparison?

Every model predicts the **same 68,508 held-out respondents**. Therefore model performance estimates are correlated.

Instead of comparing independent confidence intervals, this notebook resamples respondent indices once per bootstrap replicate and evaluates every model on the exact same resampled respondents.

**Reference model:** Logistic Regression (Model 01), prespecified because it was the literature-grounded baseline before later model results were observed.


In [ ]:
from pathlib import Path
from math import sqrt
import json
import platform
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import xgboost as xgb

from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

OUT = Path("/kaggle/working/primary_7_model_comparison")
OUT.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET = "DIABETE4"

PRIMARY_FEATURES = [
    "_AGEG5YR", "_SEX", "_RACEGR3", "_EDUCAG", "_INCOMG1", "EMPLOY1",
    "_BMI5", "GENHLTH", "PHYSHLTH", "MENTHLTH", "_TOTINDA", "_RFSMOK3",
    "SLEPTIM1", "_RFHYPE6", "_RFCHOL3", "_MICHD", "CVDSTRK3",
    "CHCKDNY2", "DIFFWALK", "PERSDOC3",
]
CATEGORICAL_FEATURES = [
    "_AGEG5YR", "_SEX", "_RACEGR3", "_EDUCAG", "_INCOMG1", "EMPLOY1",
    "GENHLTH", "_TOTINDA", "_RFSMOK3", "_RFHYPE6", "_RFCHOL3", "_MICHD",
    "CVDSTRK3", "CHCKDNY2", "DIFFWALK", "PERSDOC3",
]
NUMERIC_FEATURES = ["_BMI5", "PHYSHLTH", "MENTHLTH", "SLEPTIM1"]

print({
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "xgboost": xgb.__version__,
})


In [ ]:
xpt_candidates = glob.glob("/kaggle/input/**/LLCP2025.XPT", recursive=True)
if len(xpt_candidates) != 1:
    raise RuntimeError(f"Expected one LLCP2025.XPT, found {xpt_candidates}")

df = pd.read_sas(xpt_candidates[0], format="xport", encoding="utf-8")

def keep_codes(series, valid_codes):
    return series.where(series.isin(valid_codes), np.nan)

def recode_brfss_2025(raw):
    out = pd.DataFrame(index=raw.index)
    out["_AGEG5YR"] = keep_codes(raw["_AGEG5YR"], range(1, 14))
    out["_SEX"] = keep_codes(raw["_SEX"], [1, 2])
    out["_RACEGR3"] = keep_codes(raw["_RACEGR3"], range(1, 6))
    out["_EDUCAG"] = keep_codes(raw["_EDUCAG"], range(1, 5))
    out["_INCOMG1"] = keep_codes(raw["_INCOMG1"], range(1, 8))
    out["EMPLOY1"] = keep_codes(raw["EMPLOY1"], range(1, 9))
    out["GENHLTH"] = keep_codes(raw["GENHLTH"], range(1, 6))
    out["_TOTINDA"] = keep_codes(raw["_TOTINDA"], [1, 2])
    out["_RFSMOK3"] = keep_codes(raw["_RFSMOK3"], [1, 2])
    out["_RFHYPE6"] = keep_codes(raw["_RFHYPE6"], [1, 2])
    out["_RFCHOL3"] = keep_codes(raw["_RFCHOL3"], [1, 2])
    out["_MICHD"] = keep_codes(raw["_MICHD"], [1, 2])
    out["CVDSTRK3"] = keep_codes(raw["CVDSTRK3"], [1, 2])
    out["CHCKDNY2"] = keep_codes(raw["CHCKDNY2"], [1, 2])
    out["DIFFWALK"] = keep_codes(raw["DIFFWALK"], [1, 2])
    out["PERSDOC3"] = keep_codes(raw["PERSDOC3"], [1, 2, 3])
    out["_BMI5"] = raw["_BMI5"] / 100.0
    for col in ["PHYSHLTH", "MENTHLTH"]:
        s = raw[col].replace({88.0: 0.0, 77.0: np.nan, 99.0: np.nan})
        out[col] = s.where(s.between(0, 30), np.nan)
    sleep = raw["SLEPTIM1"].replace({77.0: np.nan, 99.0: np.nan})
    out["SLEPTIM1"] = sleep.where(sleep.between(1, 24), np.nan)
    return out[PRIMARY_FEATURES]

mask = df[TARGET].isin([1.0, 3.0])
X = recode_brfss_2025(df.loc[mask, PRIMARY_FEATURES].copy())
y = (df.loc[mask, TARGET] == 1.0).astype("int8").rename("diabetes")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

assert len(X_train) == 274_031
assert len(X_test) == 68_508
assert int(y_test.sum()) == 10_365

print("Train:", X_train.shape, "Test:", X_test.shape)


## 1. Fit the Frozen Preprocessing Once on Training Data

Because the preprocessing rules are already frozen and identical for all models, the comparison notebook fits the transformer **once on the training set only** and shares the resulting matrices.

This is mathematically equivalent to the frozen per-model pipelines, while avoiding repeated preprocessing work.

No test information enters the fit.


In [ ]:
categorical_pipeline = sklearn.pipeline.Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1.0)),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])
numeric_pipeline = sklearn.pipeline.Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
], remainder="drop")

X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

print("Transformed train:", X_train_t.shape)
print("Transformed test:", X_test_t.shape)
assert X_train_t.shape[1] == 82
assert X_test_t.shape[1] == 82


## 2. Refit the Seven Frozen Models

These are exact frozen configurations from Models 01–07.

This step exists only to create respondent-level paired predictions. No tuning is performed.


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        penalty="l2", C=1.0, solver="lbfgs", max_iter=1000, class_weight=None
    ),
    "Decision Tree": DecisionTreeClassifier(
        criterion="gini", splitter="best", max_depth=None,
        min_samples_split=2, min_samples_leaf=1, max_features=None,
        class_weight=None, ccp_alpha=0.0, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, criterion="gini", max_depth=None,
        min_samples_split=2, min_samples_leaf=1, max_features="sqrt",
        bootstrap=True, class_weight=None, ccp_alpha=0.0,
        random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        loss="log_loss", learning_rate=0.1, n_estimators=100,
        subsample=1.0, criterion="friedman_mse",
        min_samples_split=2, min_samples_leaf=1, max_depth=3,
        max_features=None, random_state=42
    ),
    "XGBoost": XGBClassifier(
        objective="binary:logistic", eval_metric="logloss", booster="gbtree",
        n_estimators=100, learning_rate=0.3, max_depth=6,
        min_child_weight=1, gamma=0.0, subsample=1.0,
        colsample_bytree=1.0, reg_alpha=0.0, reg_lambda=1.0,
        tree_method="hist", random_state=42, n_jobs=-1, verbosity=0
    ),
    "Linear SVM": LinearSVC(
        penalty="l2", loss="squared_hinge", dual="auto",
        tol=1e-4, C=1.0, fit_intercept=True, class_weight=None,
        random_state=42, max_iter=10000
    ),
    "Gaussian Naive Bayes": GaussianNB(priors=None, var_smoothing=1e-9),
}

predictions = pd.DataFrame({
    "row_index": X_test.index.to_numpy(),
    "y_true": y_test.to_numpy(),
})

dense_train = None
dense_test = None

for name, model in models.items():
    print("Fitting:", name)
    if name == "Gaussian Naive Bayes":
        if dense_train is None:
            dense_train = X_train_t.toarray() if hasattr(X_train_t, "toarray") else np.asarray(X_train_t)
            dense_test = X_test_t.toarray() if hasattr(X_test_t, "toarray") else np.asarray(X_test_t)
        model.fit(dense_train, y_train)
        score = model.predict_proba(dense_test)[:, 1]
        pred = (score >= 0.5).astype(int)
        predictions[name + "__probability"] = score
    elif name == "Linear SVM":
        model.fit(X_train_t, y_train)
        score = model.decision_function(X_test_t)
        pred = model.predict(X_test_t)
        predictions[name + "__decision"] = score
    else:
        model.fit(X_train_t, y_train)
        score = model.predict_proba(X_test_t)[:, 1]
        pred = (score >= 0.5).astype(int)
        predictions[name + "__probability"] = score

    predictions[name + "__pred"] = pred

predictions.to_csv(OUT / "test_predictions.csv.gz", index=False, compression="gzip")
print("Saved paired predictions:", predictions.shape)


## 3. Reconciliation Against Frozen Results

Before statistical comparison, the refitted models must reproduce the previously frozen point estimates closely.

If a metric differs materially, the comparison run should stop rather than silently compare a drifted implementation.


In [ ]:
frozen = {
    "Logistic Regression": {"accuracy":0.8558270566941087,"roc_auc":0.8261678122652486,"pr_auc":0.44663290841217973},
    "Decision Tree": {"accuracy":0.78615636130087,"roc_auc":0.6020699144719784,"pr_auc":0.20533779176339034},
    "Random Forest": {"accuracy":0.8510392946809132,"roc_auc":0.8044953607113304,"pr_auc":0.40246501462675927},
    "Gradient Boosting": {"accuracy":0.8563525427687277,"roc_auc":0.8260155801805384,"pr_auc":0.4483169949856431},
    "XGBoost": {"accuracy":0.8548198750510889,"roc_auc":0.8262898793557036,"pr_auc":0.44171377834099224},
    "Linear SVM": {"accuracy":0.8559876218835757,"roc_auc":0.8261580968770885,"pr_auc":0.44788405460506064},
    "Gaussian Naive Bayes": {"accuracy":0.7272873241081333,"roc_auc":0.7885646728624294,"pr_auc":0.36412201184388493},
}

def get_score(name):
    if name == "Linear SVM":
        return predictions[name + "__decision"].to_numpy()
    return predictions[name + "__probability"].to_numpy()

y_np = predictions["y_true"].to_numpy()
recon_rows = []
for name in models:
    pred = predictions[name + "__pred"].to_numpy()
    score = get_score(name)
    current = {
        "accuracy": accuracy_score(y_np, pred),
        "roc_auc": roc_auc_score(y_np, score),
        "pr_auc": average_precision_score(y_np, score),
    }
    row = {"model": name}
    for metric in ["accuracy","roc_auc","pr_auc"]:
        row[metric] = current[metric]
        row[metric + "_frozen"] = frozen[name][metric]
        row[metric + "_abs_diff"] = abs(current[metric] - frozen[name][metric])
        if row[metric + "_abs_diff"] > 0.001:
            raise RuntimeError(f"{name} {metric} drifted by {row[metric + '_abs_diff']:.6f}")
    recon_rows.append(row)

reconciliation = pd.DataFrame(recon_rows)
display(reconciliation)
reconciliation.to_csv(OUT / "reconciliation.csv", index=False)


## 4. Point Metrics from Paired Predictions

Threshold metrics are computed using each model's frozen native threshold:
- 0.5 for probability models;
- zero margin for Linear SVM.

Brier score is reported only for models with actual probabilities.


In [ ]:
metric_rows = []
for name in models:
    pred = predictions[name + "__pred"].to_numpy()
    score = get_score(name)
    tn, fp, fn, tp = confusion_matrix(y_np, pred, labels=[0,1]).ravel()
    row = {
        "model": name,
        "accuracy": accuracy_score(y_np, pred),
        "precision": precision_score(y_np, pred, zero_division=0),
        "recall": recall_score(y_np, pred),
        "specificity": tn/(tn+fp),
        "f1": f1_score(y_np, pred),
        "roc_auc": roc_auc_score(y_np, score),
        "pr_auc": average_precision_score(y_np, score),
        "brier_score": np.nan,
        "tn":tn,"fp":fp,"fn":fn,"tp":tp,
    }
    if name != "Linear SVM":
        row["brier_score"] = brier_score_loss(y_np, score)
    metric_rows.append(row)

point_metrics = pd.DataFrame(metric_rows)
display(point_metrics)
point_metrics.to_csv(OUT / "point_metrics.csv", index=False)


## 5. Paired Stratified Bootstrap Against Logistic Regression

Logistic Regression is the prespecified reference.

For every bootstrap replicate:
1. resample positive respondents with replacement;
2. resample negative respondents with replacement;
3. concatenate them;
4. apply the exact same sampled respondent indices to every model;
5. compute metrics and subtract the Logistic Regression metric.

The resulting confidence intervals refer directly to **model-minus-baseline differences**.

No p-value is required for the primary interpretation: if the 95% paired difference interval spans zero, the observed numerical difference should be treated cautiously.


In [ ]:
N_BOOT = 500
BOOT_SEED = 20251010
rng = np.random.default_rng(BOOT_SEED)

pos_idx = np.flatnonzero(y_np == 1)
neg_idx = np.flatnonzero(y_np == 0)
metric_names = ["accuracy","recall","specificity","f1","roc_auc","pr_auc"]

def metrics_for(name, idx):
    yb = y_np[idx]
    pred = predictions[name + "__pred"].to_numpy()[idx]
    score = get_score(name)[idx]
    tn, fp, fn, tp = confusion_matrix(yb, pred, labels=[0,1]).ravel()
    return {
        "accuracy": accuracy_score(yb,pred),
        "recall": recall_score(yb,pred),
        "specificity": tn/(tn+fp),
        "f1": f1_score(yb,pred),
        "roc_auc": roc_auc_score(yb,score),
        "pr_auc": average_precision_score(yb,score),
    }

boot_rows = []
for b in range(N_BOOT):
    sample_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True)
    sample_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
    idx = np.concatenate([sample_pos, sample_neg])
    rng.shuffle(idx)

    values = {name: metrics_for(name, idx) for name in models}
    ref = values["Logistic Regression"]
    for name in models:
        if name == "Logistic Regression":
            continue
        row = {"replicate":b+1,"model":name}
        for metric in metric_names:
            row["delta_" + metric] = values[name][metric] - ref[metric]
        if name != "Linear SVM":
            p_model = get_score(name)[idx]
            p_ref = get_score("Logistic Regression")[idx]
            row["delta_brier_score"] = (
                brier_score_loss(y_np[idx], p_model) - brier_score_loss(y_np[idx], p_ref)
            )
        else:
            row["delta_brier_score"] = np.nan
        boot_rows.append(row)

boot = pd.DataFrame(boot_rows)
boot.to_csv(OUT / "paired_bootstrap_replicates_vs_logistic.csv", index=False)

point_lookup = point_metrics.set_index("model")
summary_rows = []
for name in models:
    if name == "Logistic Regression":
        continue
    subset = boot[boot.model == name]
    for metric in metric_names + ["brier_score"]:
        col = "delta_" + metric
        if subset[col].isna().all():
            continue
        point_delta = point_lookup.loc[name,metric] - point_lookup.loc["Logistic Regression",metric]
        vals = subset[col].dropna().to_numpy()
        summary_rows.append({
            "model":name,
            "metric":metric,
            "point_delta_vs_logistic":point_delta,
            "bootstrap_mean_delta":float(vals.mean()),
            "ci95_lower":float(np.quantile(vals,0.025)),
            "ci95_upper":float(np.quantile(vals,0.975)),
        })

paired_summary = pd.DataFrame(summary_rows)
display(paired_summary)
paired_summary.to_csv(OUT / "paired_difference_summary_vs_logistic.csv", index=False)


## 6. Comparison Figures

The plots below visualize the same held-out respondents for every model.

They are descriptive and do not alter the frozen models.


In [ ]:
plt.figure(figsize=(7,6))
for name in models:
    fpr,tpr,_=roc_curve(y_np,get_score(name))
    auc=roc_auc_score(y_np,get_score(name))
    plt.plot(fpr,tpr,label=f"{name} ({auc:.3f})")
plt.plot([0,1],[0,1],linestyle="--",label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Primary Seven-Model ROC Curves")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT/"all_models_roc.png",dpi=180)
plt.show()

plt.figure(figsize=(7,6))
for name in models:
    precision,recall,_=precision_recall_curve(y_np,get_score(name))
    ap=average_precision_score(y_np,get_score(name))
    plt.plot(recall,precision,label=f"{name} ({ap:.3f})")
plt.axhline(y=y_np.mean(),linestyle="--",label=f"Prevalence ({y_np.mean():.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Primary Seven-Model Precision–Recall Curves")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT/"all_models_pr.png",dpi=180)
plt.show()

for metric in ["roc_auc","pr_auc"]:
    plot_df=paired_summary[paired_summary.metric==metric].copy()
    x=np.arange(len(plot_df))
    yv=plot_df["point_delta_vs_logistic"].to_numpy()
    lo=yv-plot_df["ci95_lower"].to_numpy()
    hi=plot_df["ci95_upper"].to_numpy()-yv
    plt.figure(figsize=(8,5))
    plt.errorbar(x,yv,yerr=np.vstack([lo,hi]),fmt="o",capsize=4)
    plt.axhline(0,linestyle="--")
    plt.xticks(x,plot_df.model,rotation=35,ha="right")
    plt.ylabel(f"Δ {metric} vs Logistic Regression")
    plt.title(f"Paired Bootstrap Difference — {metric}")
    plt.tight_layout()
    plt.savefig(OUT/f"paired_delta_{metric}.png",dpi=180)
    plt.show()


## 7. Interpretation Rule

This comparison is deliberately conservative.

A numerically larger point estimate does **not** automatically establish a practically meaningful difference.

Primary interpretation:
- focus on ROC-AUC and PR-AUC for ranking/discrimination under class imbalance;
- inspect paired bootstrap difference intervals versus Logistic Regression;
- treat threshold metrics as operating-point behavior rather than intrinsic ranking quality;
- treat Brier score only for models that produce actual probabilities;
- do not tune thresholds or hyperparameters after this comparison.

This notebook closes the primary model-comparison phase.
